In [3]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

dispositivos = ['geradores','alternadores']
requisitos = ['fiacao','teste']

valores_maximo = {
    requisitos[0]:260,
    requisitos[1]:140,
}

informacoes={
    dispositivos[0] :{
        requisitos[0]:2,
        requisitos[1]:1,

    },
    dispositivos[1] :{
        requisitos[0]:3,
        requisitos[1]:2,
    }
}

lucro = {
    dispositivos[0]: 250,
    dispositivos[1]: 150,
}



In [ ]:
model = pyo.ConcreteModel()

model.dispositivos = pyo.Set(initialize = dispositivos)
model.requisitos = pyo.Set(initialize = requisitos)
model.x = pyo.Var(model.dispositivos, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(model.x[d]*lucro[d] for d in model.dispositivos)
model.objetivo = pyo.Objective(rule=obj, sense=pyo.maximize)

def restricoes(model, r):
    return sum(model.x[d]*informacoes[d][r] for d in model.dispositivos)<=valores_maximo[r]
model.requisitos = pyo.Constraint(model.requisitos, rule=restricoes)

def qt_minima(model,d):
    return (model.x[d]) >= 20
model.qt_min = pyo.Constraint(model.dispositivos, rule=qt_minima)



'pyomo.core.base.set.OrderedScalarSet'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


In [25]:
model.pprint()


1 Set Declarations
    dispositivos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'geradores', 'alternadores'}

1 Var Declarations
    x : Size=2, Index=dispositivos
        Key          : Lower : Value : Upper : Fixed : Stale : Domain
        alternadores :     0 :  None :  None : False :  True : NonNegativeIntegers
           geradores :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 250*x[geradores] + 150*x[alternadores]

1 Constraint Declarations
    requisitos : Size=2, Index=requisitos, Active=True
        Key    : Lower : Body                               : Upper : Active
        fiacao :  -Inf : 2*x[geradores] + 3*x[alternadores] : 260.0 :   True
         teste :  -Inf :   x[geradores] + 2*x[alternadores] : 140.0 :   True

4 Declarat

In [26]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)



Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmpp7og86ev.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpi13d_qjt.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpi13d_qjt.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       2  [Less: 2]
  Nonzeros           :       4
  RHS nonzeros       :       2

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 150.0000       

In [27]:
for m in model.x:
    print(f"Quantidade do dispositivo: {m}= {pyo.value(model.x[m])}")

print(f"Lucro: {pyo.value(model.objetivo)}")

Quantidade do dispositivo: geradores= 130.0
Quantidade do dispositivo: alternadores= -0.0
Lucro: 32500.0
